# S669 sequence-budget calibration on a free GPU

Reproduces **thread 2**'s foldability-margin calibration with a *large* ESM2 model — the fair number the repo's author could not reach on a 2-core CPU.

**Runtime → Change runtime type → GPU (T4 is fine)**, then Runtime → Run all.

It scores every S669 variant with ESM2 masked-marginal log-odds and correlates against the *measured* experimental ΔΔG. On an 8M model the author got Spearman ~0.19-0.21; 35M already doubled it to ~0.36. This runs 650M and reports the real number.

In [ ]:
!pip -q install fair-esm scipy >/dev/null
import urllib.request, os
os.makedirs('ddg', exist_ok=True)
for f in ['S669.tsv','S669.fasta']:
    urllib.request.urlretrieve(f'https://ddgemb.biocomp.unibo.it/static/ddgemb/data/{f}', f'ddg/{f}')
print('downloaded S669')

In [ ]:
import torch, esm, numpy as np
from scipy.stats import spearmanr
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

# ---- pick the model: bump to esm2_t36_3B_UR50D if you have the VRAM ----
MODEL = 'esm2_t33_650M_UR50D'
MAXLEN = 1022

def load_fasta(p):
    d={};name=None;buf=[]
    for l in open(p):
        l=l.rstrip()
        if l.startswith('>'):
            if name:d[name]=''.join(buf)
            name=l[1:].strip();buf=[]
        else:buf.append(l)
    if name:d[name]=''.join(buf)
    return d
seqs = load_fasta('ddg/S669.fasta')
rows = [r.split('\t') for r in open('ddg/S669.tsv').read().splitlines()[1:] if len(r.split('\t'))>=5]
print(len(rows),'mutations,',len(seqs),'sequences')

In [ ]:
model, alphabet = getattr(esm.pretrained, MODEL)()
model = model.to(DEVICE).eval()
bc = alphabet.get_batch_converter(); mask_idx = alphabet.mask_idx
tok = {a: alphabet.get_idx(a) for a in 'ACDEFGHIKLMNPQRSTVWY'}

# masked-marginal: for each sequence, mask each needed position (batched) and read log-probs
byseq = {}
for r in rows:
    pid,wt,pos,mt,ddg = r[0],r[1],int(r[2]),r[3],float(r[4])
    if pid not in seqs or len(seqs[pid])>MAXLEN: continue
    s = seqs[pid]
    if pos<1 or pos>len(s) or s[pos-1]!=wt or wt not in tok or mt not in tok: continue
    byseq.setdefault(pid, []).append((wt,pos,mt,ddg))

scores=[]; ddgs=[]
for n,(pid,muts) in enumerate(byseq.items()):
    s = seqs[pid]; _,_,toks = bc([('x',s)]); toks = toks.to(DEVICE)
    positions = sorted({p for _,p,_,_ in muts}); cache={}
    for i in range(0, len(positions), 8):
        chunk = positions[i:i+8]
        batch = toks.repeat(len(chunk),1).clone()
        for r_,pos in enumerate(chunk): batch[r_,pos]=mask_idx
        with torch.no_grad():
            lp = torch.log_softmax(model(batch)['logits'], dim=-1)
        for r_,pos in enumerate(chunk): cache[pos]=lp[r_,pos].cpu()
    for wt,pos,mt,ddg in muts:
        scores.append((cache[pos][tok[mt]]-cache[pos][tok[wt]]).item()); ddgs.append(ddg)
    if (n+1)%15==0: print(f'{n+1}/{len(byseq)} sequences')
scores=np.array(scores); ddgs=np.array(ddgs)
print('scored', len(scores), 'variants')

In [ ]:
import matplotlib.pyplot as plt
rho,pv = spearmanr(scores, ddgs)
tol=ddgs>-0.5; dest=ddgs<-1.0
av=np.concatenate([scores[tol],scores[dest]]); lab=np.concatenate([np.ones(tol.sum()),np.zeros(dest.sum())])
o=np.argsort(av); rk=np.empty(len(av)); rk[o]=np.arange(1,len(av)+1)
auc=(rk[lab==1].sum()-lab.sum()*(lab.sum()+1)/2)/(lab.sum()*(lab==0).sum())
print(f'{MODEL}: Spearman {rho:+.3f} (p={pv:.1e}),  AUC {auc:.3f},  n={len(scores)}')
plt.figure(figsize=(6,5)); plt.scatter(ddgs,scores,s=12,alpha=0.5)
z=np.polyfit(ddgs,scores,1); xs=np.linspace(ddgs.min(),ddgs.max(),40)
plt.plot(xs,np.polyval(z,xs),'r'); plt.axvline(0,ls=':',c='gray')
plt.xlabel('measured ΔΔG (kcal/mol)  negative=destabilizing'); plt.ylabel('ESM variant score')
plt.title(f'{MODEL} vs S669  —  Spearman {rho:+.2f}'); plt.tight_layout(); plt.show()